In [ ]:
%%capture
# Install the Indian Factor Library (no-op if already installed)
import sys
!{sys.executable} -m pip install indiafactorlibrary

In [ ]:
# =============================================================================
# Notebook : 00 — Indian Market Puzzles
# Series   : pyIndiaFactorInvesting
# Purpose  : Five empirical observations that motivate the rest of the series.
#            No theory — only data.
# Data     : data/sample/nifty_monthly_returns.csv
#            data/sample/fund_returns.csv
# =============================================================================

# Five Puzzles in Indian Market Data

This notebook is the entry point for **pyIndiaFactorInvesting** — an open-source
educational series on factor investing in India.

We start with observation, not theory.  Five charts. Five patterns that resist
easy explanation.  No answers yet — just questions worth pursuing.

---

**Prerequisites** 
No prior knowledge of factor models is required.  A basic familiarity with
returns and charts is sufficient.

**After this notebook you will be able to** 
- Describe the size, quality, momentum, and volatility anomalies as they appear in Indian data 
- Articulate why these patterns are surprising relative to conventional finance wisdom 
- Name the empirical questions the rest of this series will answer

In [ ]:
import sys
from pathlib import Path

# Make src/ importable both locally and on Colab
sys.path.insert(0, str(Path('..').resolve()))

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data import load_nifty_sample, load_fund_sample
from src.visualise import plot_cumulative_returns

# Event dates reused across puzzles
EVENTS = {
    'Demonetisation\n(Nov 2016)': '2016-11-30',
    'IL&FS\n(Sep 2018)':          '2018-09-30',
    'COVID\n(Mar 2020)':          '2020-03-31',
}

---
## Puzzle 1 — Does size matter?

Conventional wisdom says smaller companies should deliver higher long-run
returns to compensate for their extra risk.  Let's look at the Indian evidence.

In [ ]:
nifty = load_nifty_sample()
size_df = nifty[['nifty100', 'nifty_midcap150', 'nifty_smallcap250']].rename(columns={
    'nifty100':          'Nifty 100 (Large Cap)',
    'nifty_midcap150':   'Nifty Midcap 150',
    'nifty_smallcap250': 'Nifty Smallcap 250',
})

ax = plot_cumulative_returns(size_df, title='Puzzle 1: Does Size Matter? (Jan 2015 – Dec 2020)')

# Annotate macro events
for label, date in EVENTS.items():
    ax.axvline(pd.Timestamp(date), color='dimgrey', linewidth=0.8, linestyle='--')
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.95, label,
            fontsize=7, ha='center', va='top', color='dimgrey')

plt.show()

**Observation** 
Smallcap outpaced large caps through 2017 — consistent with the size premium
predicted by standard theory.  Then, from mid-2018, it collapsed and never
recovered within this window while large caps continued higher.

**The puzzle:** If the size premium is a reward for risk, why did the risk
materialise so severely *and* so persistently in 2018?  And why did it
coincide precisely with the IL&FS credit crisis rather than a broad market
event?

---
## Puzzle 2 — Quality outperforms when it matters most

"Quality" stocks — those with high return on equity, low leverage, and stable
earnings — are often said to be overpriced because everyone wants them.
Let's see whether that held in India.

In [ ]:
quality_df = nifty[['nifty100', 'nifty100_quality30']].rename(columns={
    'nifty100':            'Nifty 100 (Benchmark)',
    'nifty100_quality30':  'Nifty 100 Quality 30',
})

ax = plot_cumulative_returns(quality_df, title='Puzzle 2: Quality Outperforms When It Matters Most')

# Highlight the IL&FS stress period
ax.axvspan(pd.Timestamp('2018-07-31'), pd.Timestamp('2019-03-31'),
           alpha=0.12, color='crimson', label='IL&FS stress period')
ax.legend(fontsize=8, frameon=False)

plt.show()

**Observation** 
Quality tracked the benchmark closely for most of the period, then diverged
sharply during the IL&FS stress window (shaded), preserving capital while the
broad market fell.

**The puzzle:** Standard asset pricing says higher return = higher risk.  But
quality delivered *higher* returns with *lower* drawdowns during stress.  Is
that alpha?  Or is quality simply capturing a dimension of risk that standard
models miss?

---
## Puzzle 3 — Momentum: spectacular until it isn't

Momentum — buying recent winners and selling recent losers — is one of the
most documented anomalies in global markets.  Indian data tells a familiar
story, with a twist.

In [ ]:
momentum_df = nifty[['nifty100', 'nifty200_momentum30']].rename(columns={
    'nifty100':             'Nifty 100 (Benchmark)',
    'nifty200_momentum30':  'Nifty 200 Momentum 30',
})

ax = plot_cumulative_returns(momentum_df, title="Puzzle 3: Momentum — Spectacular Until It Isn't")

# Mark the momentum crash dates
crash_events = {
    'Momentum\ncrash 1\n(Oct 2018)': '2018-10-31',
    'Momentum\ncrash 2\n(Jul 2019)': '2019-07-31',
}
for label, date in crash_events.items():
    ax.axvline(pd.Timestamp(date), color='firebrick', linewidth=0.8, linestyle=':')
    ax.text(pd.Timestamp(date), ax.get_ylim()[0] * 0.85, label,
            fontsize=7, ha='center', va='bottom', color='firebrick')

plt.show()

**Observation** 
Momentum built a substantial lead over the benchmark — then gave it all back
in two sharp reversals (marked).  The pattern repeats: long stretches of
outperformance punctuated by sudden, severe drawdowns.

**The puzzle:** If momentum crashes are predictable features of the strategy,
why doesn't arbitrage eliminate the premium?  And what triggers the reversals
in India specifically?

---
## Puzzle 4 — Fund returns: same category, very different journeys

Three funds.  Same Indian equity mandate.  Same time period.  Same benchmark.
How different can their return paths be?

In [ ]:
funds = load_fund_sample()

ax = plot_cumulative_returns(funds, title='Puzzle 4: Same Category, Very Different Journeys')
plt.show()

**Observation** 
Fund A, B, and C all operate within the same broad mandate, yet their
cumulative return paths diverge substantially.  Fund A accumulated the most
wealth but with pronounced swings.  Fund B barely moved.  Fund C oscillated
wildly around the zero line.

**The puzzle:** If all three face the same market, what explains the
dispersion?  Is it manager skill, factor tilts, or something else entirely?

---
## Puzzle 5 — Risk and return: the Indian evidence

Modern portfolio theory's central claim: investors must accept higher risk
to earn higher returns.  Does that hold for these three funds?

In [ ]:
_BASE_FONT = 10
_FIG_DPI   = 120

# Rolling 12-month annualised volatility
rolling_vol = funds.rolling(12).std() * (12 ** 0.5)

# Cumulative returns
cum_ret = (1 + funds).cumprod() - 1

fund_labels = funds.columns.tolist()   # ['Fund_A', 'Fund_B', 'Fund_C']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(3, 1, figsize=(9, 9), dpi=_FIG_DPI, sharex=True)
fig.suptitle('Puzzle 5: Risk and Return — the Indian Evidence',
             fontsize=_BASE_FONT + 1, fontweight='bold')

for i, (fund, color) in enumerate(zip(fund_labels, colors)):
    ax_vol = axes[i]
    ax_ret = ax_vol.twinx()

    # Left axis — rolling volatility
    ax_vol.fill_between(rolling_vol.index, rolling_vol[fund],
                        alpha=0.30, color=color)
    ax_vol.plot(rolling_vol.index, rolling_vol[fund],
                color=color, linewidth=1.2, label='12M Ann. Vol')
    ax_vol.set_ylabel('Ann. Vol', fontsize=_BASE_FONT - 1, color=color)
    ax_vol.tick_params(axis='y', labelcolor=color, labelsize=_BASE_FONT - 1)
    ax_vol.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax_vol.spines['top'].set_visible(False)

    # Right axis — cumulative return
    ax_ret.plot(cum_ret.index, cum_ret[fund],
                color=color, linewidth=1.4, linestyle='--', label='Cum. Return')
    ax_ret.axhline(0, color='black', linewidth=0.5, linestyle=':')
    ax_ret.set_ylabel('Cum. Return', fontsize=_BASE_FONT - 1)
    ax_ret.tick_params(axis='y', labelsize=_BASE_FONT - 1)
    ax_ret.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
    ax_ret.spines['top'].set_visible(False)

    # Fund label
    ax_vol.set_title(fund, fontsize=_BASE_FONT, loc='left', pad=3)

    # Combined legend on first subplot only
    if i == 0:
        lines1, labels1 = ax_vol.get_legend_handles_labels()
        lines2, labels2 = ax_ret.get_legend_handles_labels()
        ax_vol.legend(lines1 + lines2, labels1 + labels2,
                      fontsize=_BASE_FONT - 2, frameon=False, loc='upper left')

axes[-1].set_xlabel('Date', fontsize=_BASE_FONT)

# Source annotation
fig.text(0.99, 0.01, 'Source: Invespar Indian Factor Library',
         ha='right', va='bottom', fontsize=_BASE_FONT - 2, color='grey')

fig.tight_layout()
plt.show()

**Observation** 
Higher rolling volatility does not consistently accompany higher cumulative
returns.  Fund C shows elevated volatility relative to its terminal wealth.
Fund A built the largest cumulative return without sustaining the highest
volatility throughout.

**The puzzle:** If risk and return are positively linked, why do we observe
funds with high volatility but poor outcomes?  Is volatility even the right
measure of risk for Indian equity funds?

---
## Where do we go from here?

Five puzzles.  No answers yet — by design.

| Puzzle | Question |
|--------|----------|
| 1 — Size | Why did the size premium reverse so sharply in 2018 and stay down? |
| 2 — Quality | How can lower-risk stocks deliver higher returns during crises? |
| 3 — Momentum | What triggers sudden momentum crashes in India? |
| 4 — Dispersion | What explains the large spread in fund returns within a single category? |
| 5 — Volatility | Is volatility the wrong proxy for risk in Indian markets? |

The rest of this series builds the tools — factor models, regressions, risk
decomposition — needed to examine these questions rigorously.

**Next:** `01_factor_zoo.ipynb` — what is a factor and why does it persist?